# Data Pipeline: Dataset, Split & DataLoader

Builds the stratified train/val/test split and the custom PyTorch `Dataset` (with augmentation) used to feed the garment classifier. Prototyped here, final version moves to `src/data.py`.

## 1. Load Clean Dataset

In [1]:
import pandas as pd

targets_dir = "../data/datasets/paramaggarwal/fashion-product-images-small/versions/1/images"

df = pd.read_csv("../data/clean_data/clean_df.csv")
df.shape

(20436, 2)

## 2. Stratified Train/Val/Test Split

In [2]:
from sklearn.model_selection import train_test_split

# Two-step split: 80% train, then split the remaining 20% into val/test (10%/10%)
# stratify keeps class proportions consistent across splits, important given the class imbalance
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["articleType"]
)

# stratify here uses temp_df's own labels, not df's, since we're splitting temp_df now
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["articleType"]
)

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(16348, 2)
(2044, 2)
(2044, 2)


In [3]:
# Sanity check: confirms stratify actually preserved per-class proportions across splits
print("\nTRAIN DF -------------------------------\n")
print(train_df["articleType"].value_counts())
print("\nVALIDATION DF -------------------------------\n")
print(val_df["articleType"].value_counts())
print("\nTEST DF -------------------------------\n")
print(test_df["articleType"].value_counts())


TRAIN DF -------------------------------

articleType
Tshirts         5655
Shirts          2572
Casual Shoes    2277
Sports Shoes    1629
Tops            1409
Formal Shoes     510
Jeans            486
Shorts           438
Dresses          371
Track Pants      243
Sweatshirts      228
Sweaters         222
Jackets          206
Skirts           102
Name: count, dtype: int64

VALIDATION DF -------------------------------

articleType
Tshirts         707
Shirts          321
Casual Shoes    285
Sports Shoes    204
Tops            177
Formal Shoes     63
Jeans            61
Shorts           54
Dresses          47
Track Pants      30
Sweatshirts      29
Sweaters         27
Jackets          26
Skirts           13
Name: count, dtype: int64

TEST DF -------------------------------

articleType
Tshirts         707
Shirts          322
Casual Shoes    284
Sports Shoes    203
Tops            176
Formal Shoes     64
Jeans            61
Shorts           55
Dresses          46
Track Pants      31
Sweat

In [4]:
from torch.utils.data import Dataset
from PIL import Image

# Fixed list, same order every time, so train/val/test instances all
# agree on which integer maps to which articleType
categories = ["Tshirts","Shirts","Tops","Jeans","Sweatshirts","Sweaters",
      "Jackets","Dresses","Skirts","Shorts","Track Pants",
      "Casual Shoes","Sports Shoes","Formal Shoes"]

class wardroveDataset(Dataset):
    def __init__(self, dataframe, target_imgs_dir, transform):
        # DataFrame slice for this split (train_df, val_df, or test_df)
        self.dataframe = dataframe
        # Folder holding the actual .jpg files, shared across all splits
        self.target_imgs_dir = target_imgs_dir
        # torchvision transforms.Compose pipeline, different for train vs eval
        self.transform = transform

        # Translates articleType strings to the integer ids CrossEntropyLoss expects
        self.label_to_val = {label: val for val, label in enumerate(categories)}

    def __len__(self):
        # Tells the DataLoader how many samples exist, so it knows the valid idx range
        return len(self.dataframe)

    def __getitem__(self, idx):
        # idx is a position (0 to len-1)
        row = self.dataframe.iloc[idx]
        # Build the path to this row's target image using its real id
        path = f'{self.target_imgs_dir}/{row["id"]}.jpg'
        # convert("RGB") normalizes channels in case a file is grayscale or has alpha
        image = Image.open(path).convert("RGB")
        # Applies resize/augmentation/ToTensor, returns a tensor ready for the model
        image = self.transform(image)
        # String label ("Shirts") converted to the integer the loss function expects
        label_val = self.label_to_val[row["articleType"]]

        return image, label_val
        

In [5]:
from torchvision import transforms

# Augmentation only for train: flip + color jitter, applied randomly each time
# an image is loaded, so the model sees slightly different versions across epochs
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor()
])

# No augmentation for val/test: evaluation needs to be deterministic and reflect
# real, unaltered data, not artificially varied inputs
eval_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [6]:
# One Dataset instance per split, each with its own dataframe and transform
train_dataset = wardroveDataset(train_df, targets_dir, train_transform)
val_dataset = wardroveDataset(val_df, targets_dir, eval_transform)
test_dataset = wardroveDataset(test_df, targets_dir, eval_transform)

In [7]:
# Single-sample sanity check: confirms __getitem__ works end to end before
# trusting the DataLoader to batch things correctly
image, label = train_dataset[0]
print(image.shape, image.dtype)
print(label)

torch.Size([3, 224, 224]) torch.float32
12


In [ ]:
from torch.utils.data import DataLoader

# shuffle=True only for train, so batch order varies each epoch
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
# shuffle=False for val/test, order doesn't matter and stays predictable
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [ ]:
# Pull one full batch to confirm the DataLoader collates samples correctly
images, labels = next(iter(train_loader))
print(images.shape, labels.shape)